# Air Quality Prediction — Project 19
## Training-period EDA

**Phase 1 only.** Predict PM2.5 one hour ahead using historical measurements.
Required comparison: XGBoost, persistence, daily seasonal naive; Ridge adds a linear reference.

**How to use:** upload this notebook at [Google Colab](https://colab.research.google.com/),
choose a CPU runtime, and select **Runtime → Run all**. No GitHub connection is required.
Runtime and internet availability depend on your Colab session.
The source is included below, so this single notebook is sufficient to start.
Save an executed copy to your own Drive and download the outputs at the end.

**Evidence policy:** an unexecuted notebook contains no measured results. This notebook uses the
actual [UCI Beijing PM2.5 dataset](https://archive.ics.uci.edu/dataset/381/beijing+pm2+5+data),
never a silent synthetic replacement. Source: Chen (2015), DOI 10.24432/C5JS49, CC BY 4.0.


### 1. Restore the reusable source
Run this setup cell. In Colab the Files sidebar will show the package under `/content/air_prediction`.
All feature, split, and model code is readable in `src/air_prediction`; the encoded ZIP merely makes
the notebook portable without GitHub. When opened in the existing local project, local source is used.


In [ ]:
import base64, io, os, sys, zipfile
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path('/content/air_prediction')
elif (Path.cwd() / 'src' / 'air_prediction').is_dir():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / 'src' / 'air_prediction').is_dir():
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd() / 'air_prediction'

# This ZIP contains the same readable source/config/docs/tests shipped with the project.
# It contains no dataset, login credentials, trained model, or claimed result.
SOURCE_BUNDLE = "..."
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
if not (PROJECT_ROOT / 'src' / 'air_prediction').exists():
    with zipfile.ZipFile(io.BytesIO(base64.b64decode(SOURCE_BUNDLE))) as bundle:
        for member in bundle.infolist():
            target = (PROJECT_ROOT / member.filename).resolve()
            if not target.is_relative_to(PROJECT_ROOT.resolve()):
                raise ValueError('Invalid bundled source path')
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(bundle.read(member))
os.environ.setdefault('MPLCONFIGDIR', str(PROJECT_ROOT / '.cache' / 'matplotlib'))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
print('Project source:', PROJECT_ROOT)
print('Colab runtime:', IN_COLAB)


In [ ]:
import subprocess
INSTALL_DEPENDENCIES = IN_COLAB
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                    '-r', str(PROJECT_ROOT / 'requirements.txt'), 'pytest>=8,<9'], check=True)
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display
from air_prediction.config import load_config
from air_prediction.data import FILENAME, download_dataset, load_dataset
from air_prediction.eda import run_eda
print('numpy:', np.__version__, 'pandas:', pd.__version__)


### 2. Fix the experiment settings before viewing test results
Forecast origin = t, target = t+1, using Beijing local time. Initial training: 2010–2012.
Parameter selection: 2013. Final rolling evaluation: 2014. Refits occur quarterly; observed histories
update hourly. The EDA preview uses only initial training dates. Keep the final holdout for evaluation.


In [ ]:
CONFIG_PATH = PROJECT_ROOT / 'configs' / 'phase1.toml'
DATA_PATH = PROJECT_ROOT / 'data' / 'raw' / FILENAME
OUTPUT_ROOT = PROJECT_ROOT / 'artifacts' / 'phase1'
EDA_DIR = PROJECT_ROOT / 'artifacts' / 'eda_preview'
config = load_config(CONFIG_PATH)
display(config.to_dict())


In [ ]:
if not DATA_PATH.is_file():
    download_dataset(DATA_PATH)
dataset = load_dataset(DATA_PATH)
display(Markdown('**Data source:** ' + dataset.source))
display(pd.Series({k: v for k, v in dataset.audit.items()
                   if k not in {'provenance', 'missing_by_column'}}, name='archive audit'))
display(pd.Series(dataset.audit['missing_by_column'], name='Archive-wide missing count'))
display(dataset.frame.head())


### 3. Training EDA and quality checks
The full-archive audit above checks timestamps and completeness; the plots and statistics below use
dates **strictly before 2013**. Distinguish a missing measurement from a true zero. Preserve valid
pollution peaks; the IQR rule is descriptive, not an automatic deletion rule.


In [ ]:
run_eda(dataset, EDA_DIR, config.validation_start)
display(pd.read_csv(EDA_DIR / 'missingness.csv', index_col=0))
display(pd.read_csv(EDA_DIR / 'descriptive_statistics.csv', index_col=0))
import json
display(json.loads((EDA_DIR / 'distribution_summary.json').read_text()))


**Timeline and missingness:** identify changes over time, lengthy gaps, and variables whose
completeness changes. Describe actual observations after execution; do not infer them from this prompt.


In [ ]:
display(Image(filename=str(EDA_DIR / 'timeline_missingness.png')))


**Distribution and calendar:** compare mean and median, tail behavior, and hourly/monthly variation.
Consider whether peaks are rare but important and whether temporal patterns motivate the chosen lags.


In [ ]:
display(Image(filename=str(EDA_DIR / 'distribution_seasonality.png')))
display(Image(filename=str(EDA_DIR / 'calendar_profiles.png')))


**Correlation and autocorrelation:** association does not establish causation. ACF uses the complete
hourly grid so missing observations do not compress time. Examine lag 1, the daily scale, and the weekly scale.


In [ ]:
display(pd.read_csv(EDA_DIR / 'correlations.csv', index_col=0))
display(pd.read_csv(EDA_DIR / 'autocorrelation.csv', index_col=0))
display(Image(filename=str(EDA_DIR / 'correlation_autocorrelation.png')))


: 

### 4. Trace the causal features and baseline alignment
At t we know y(t), so lag zero is valid. Trailing windows end at t. With horizon one,
daily seasonal naive uses y(t−23), which is 24 hours before the target t+1.
No evaluation target is imputed. Weather is available only through t.


In [ ]:
from air_prediction.features import build_features, common_evaluation_mask
supervised = build_features(dataset.frame, config)
eligible_training = common_evaluation_mask(supervised) & (supervised.target_time < pd.Timestamp(config.validation_start))
trace = supervised.X.loc[eligible_training, ['pm25_lag_0', 'pm25_lag_24', 'pm25_mean_24', 'weather_TEMP_lag_0']].head()
trace['target_time'] = supervised.target_time.reindex(trace.index)
trace['actual_future_target'] = supervised.y.reindex(trace.index)
trace = trace.join(supervised.baselines)
display(trace)
print('Input features before one-hot encoding:', supervised.X.shape[1])
print('Feature source:', PROJECT_ROOT / 'src' / 'air_prediction' / 'features.py')


### Download and finish the Phase 1 submission
The ZIP includes readable source, the input CSV/provenance, measured outputs, the IEEE report source,
presentation source, and viva/contribution guides. Also save this executed notebook to your own Drive.
Fill the five real team members, interpret actual EDA/results, and compile/proofread the report and slides.
Their templates are intentionally honest about any results that have not yet been generated.

Dataset: CC BY 4.0; cite UCI dataset DOI 10.24432/C5JS49. See `docs/literature_review.md` for primary sources.


In [ ]:
export_path = PROJECT_ROOT.parent / 'air_prediction_phase1_outputs.zip'
excluded = {'__pycache__', '.cache', '.pytest_cache', '.git', '.venv', '.ipynb_checkpoints'}
with zipfile.ZipFile(export_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(PROJECT_ROOT.rglob('*')):
        relative = path.relative_to(PROJECT_ROOT)
        if path.is_file() and not (set(relative.parts) & excluded) and path.suffix not in {'.pyc', '.pyo'}:
            archive.write(path, relative.as_posix())
print('Saved:', export_path)
if IN_COLAB:
    from google.colab import files
    files.download(str(export_path))
else:
    display(Markdown('Output archive: `' + str(export_path) + '`'))
